# Transaction Foundation Model on Ray — Part 3: Tokenize with NVIDIA's tokenizer

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: longer — encodes the whole train split)

---

In Part 2 we regenerated NVIDIA's temporal split. Now we **tokenize with NVIDIA's own `FinancialTabularTokenizer`** (vendored into `src/nvidia_tokenizer/`) and build the **pretraining corpus**: each card's transactions become a token sequence, packed into 4096-token windows for the causal decoder to learn from in Part 4. Using NVIDIA's tokenizer (not a reimplementation) is what makes our foundation model match/beat theirs.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                                        # same knob as Part 2; mini = tiny & fast
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## NVIDIA's transaction tokenizer

We use NVIDIA's `FinancialTabularTokenizer` (Apache-2.0, vendored verbatim into `src/nvidia_tokenizer/`). Each transaction is turned into a small group of field tokens drawn from a **shared vocabulary of 6,251 tokens**:

- **Merchant** → hashed into 2,000 buckets (open-ended, so hashing keeps the vocab bounded).
- **Category hierarchy** → MCC mapped through a coarse→fine category structure.
- **Temporal encoding** → time-of-day / day / month plus the *gap since the previous transaction* (the bursty inter-transaction timing we saw in Part 2).
- **Amount** → binned (the log-scale spread from Part 2).

A card's history becomes a single stream — `<bos> [txn₁ tokens] <sep> [txn₂ tokens] <sep> … <eos>` — packed into fixed **4096-token** sequences (~315 transactions each, NVIDIA's chunking). That flat stream is exactly what the Llama causal decoder pretrains on by next-token prediction (Part 4).

## Build the pretraining corpus

We build the corpus from the temporal **train** split (Part 2). For each card we sort its transactions in time order, pack every `chunk` of them (~315 at full) into one `<bos> … <sep> … <eos>` sequence, and encode to `seq_len` tokens. The result is three arrays the pretrainer reads directly:

- `ids.npy` — `(n_sequences, seq_len)` int32 token ids (right-padded),
- `attn.npy` — the attention mask (1 = real token, 0 = pad),
- `vocab.json` — vocab size + pad id.

This runs as one cuDF/GPU job in `src/nvcorpus.py` (mirroring NVIDIA's NB02). The `mini` scale caps the number of sequences for a fast smoke run; `full` encodes the whole train split.

In [ ]:
from src.nvcorpus import build_corpus

tk = cfg["tokenize"]
seq_len = tk["seq_len"]                        # tokens per pretrain sequence (full 4096 = NVIDIA)
chunk = max(1, seq_len // 13)                  # ~13 tokens/txn → transactions packed per sequence
max_seq = tk.get("max_pretrain_windows")       # cap #sequences (mini/CI); None = every sequence

train_parquet = os.path.join(paths["nvsplit"], "train.parquet")

# Build the pretraining corpus with NVIDIA's tokenizer (src/nvcorpus.py, a cuDF/GPU task):
# group each card's transactions in time order, pack `chunk` of them into one
# "<bos> t1 <sep> t2 … <eos>" sequence, encode to `seq_len` tokens → ids.npy/attn.npy/vocab.json.
# Re-runs reuse the cached corpus.
if not os.path.exists(os.path.join(paths["nvcorpus"], "ids.npy")):
    meta = build_corpus(train_parquet, paths["nvcorpus"],
                        seq_len=seq_len, chunk=chunk, max_seq=max_seq)
    print(json.dumps(meta, indent=2))
else:
    print("corpus already built at", paths["nvcorpus"])

## The tokenized corpus

Let's look at what we built: the number of sequences and the fraction of real (non-pad) tokens, then decode the first sequence back to readable tokens using the same tokenizer. Each transaction's field tokens appear in order, framed by `<bos>` / `<sep>` / `<eos>` (merchant/MCC are hashed, so they read as bucket ids).

In [ ]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"), mmap_mode="r")
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"), mmap_mode="r")
with open(os.path.join(paths["nvcorpus"], "vocab.json")) as f:
    vinfo = json.load(f)

print(f"{ids.shape[0]:,} pretrain sequences × {ids.shape[1]} tokens  "
      f"(vocab {vinfo['vocab_size']:,}, pad id {vinfo['pad']})")
print(f"real-token fraction: {float(np.asarray(attn).mean()):.3f}\n")

# Decode sequence 0 back to readable tokens (drop right-padding).
tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
row = np.asarray(ids[0])
real = [int(t) for t in row if int(t) != vinfo["pad"]]
decoded = tok.decode(real).split()
print(f"sequence 0 — {len(real)} real tokens; first ~2 transactions:")
print("  " + " ".join(decoded[:32]))

The vocabulary is fixed at **6,251 tokens**, so the model knows its size up front — no data scan, no cold-start. Special tokens (`<pad>`/`<bos>`/`<eos>`/`<sep>`/`<unk>`) come first, then each field owns a disjoint block of ids (hashed merchant buckets, category hierarchy, temporal fields, amount bins).

In [ ]:
from src.nvidia_tokenizer import FinancialTabularTokenizer

tok = FinancialTabularTokenizer(merchant_hash_size=2000, category_hierarchy=True,
                                temporal_encoding=True)
vocab = tok.vocab
specials = [t for t in ("<pad>", "<bos>", "<eos>", "<sep>", "<unk>") if t in vocab]
print(f"shared vocabulary: {tok.vocab_size:,} tokens  ({len(specials)} special + per-field blocks)")
print(f"specials: {specials}\n")

# Group the vocab tokens by their field prefix to show the per-field block sizes.
from collections import Counter
prefixes = Counter(t.split("_")[0] for t in vocab if t not in specials and "_" in t)
print("per-field block size in the shared vocab:")
for name, size in sorted(prefixes.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<12} {size:>6,}")

## Takeaways

**The tokenizer**
- We use NVIDIA's own `FinancialTabularTokenizer` (vendored, Apache-2.0) — merchant hashing + category hierarchy + temporal encoding + amount binning, fixed vocab **6,251**. Using their exact tokenizer (not a reimplementation) is what lets our foundation model reach their embedding quality.
- Each card's history → a `<bos> … <sep> … <eos>` token stream, packed into 4096-token sequences (~315 transactions), the input to next-token pretraining.

**Ray**
- The corpus build runs as a single cuDF/GPU task (`src/nvcorpus.py`); the same code encodes a tiny `mini` corpus or the full ~19.5M-transaction train split — only the config changes.

---

## Next

**Part 4 — Pretrain with Ray Train**: next-token prediction over these 4096-token sequences with a Llama causal decoder, plain PyTorch wrapped by Ray Train for DDP, sharding, and checkpointing — the same code from 1 worker to N GPUs.